# KLDM Notebook

This notebook gives a small end-to-end KLDM-style training example using the real code in this repo.

We will:
- load a real `DNGTask` batch from MP-20
- diffuse lattice `l` with the VP diffusion
- diffuse atom features `a` with the VP diffusion
- diffuse velocity / positions with the trivialised diffusion class
- pass the noisy variables into `CSPVNet`
- train on a simple sum of the three score-matching losses

This is still a **simple educational notebook**. It uses the current simplified `TrivialisedDiffusion` implementation from the repo, not the full final KLDM paper pipeline.

## Theory

For one crystal graph we sample a diffusion time `t` uniformly in `[eps, 1]`.

Then we build noisy training inputs:

- lattice:
  \[
  l_t = \alpha(t) l_0 + \sigma(t) \epsilon_l
  \]
- atom features:
  \[
  a_t = \alpha(t) a_0 + \sigma(t) \epsilon_a
  \]
- trivialised momentum part:
  \[
  v_t, f_t \sim p_{t|0}(f_t, v_t \mid f_0, v_0)
  \]

The network sees `(t, f_t, v_t, l_t, a_t)` and predicts three outputs:
- a velocity head `out["v"]`
- `out["l"]`
- `out["h"]`

For the velocity part we follow the KLDM training recipe more closely:

1. the network predicts an intermediate velocity quantity
2. we convert it into the actual score with the Eq. (19)-style transform
3. we compare that score to the target from the TDM forward process

For lattice and atom features we keep the same DSM form used in the continuous notebook.

In [3]:
import sys
from pathlib import Path

import torch

repo_root = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'src' / 'kldm').is_dir())
src_root = repo_root / 'src'
notebook_dir = repo_root / 'src' / 'kldm'
sys.path = [p for p in sys.path if Path(p or '.').resolve() != notebook_dir]
sys.path.insert(0, str(src_root))

from kldm.data import DNGTask, resolve_data_root
from kldm.distribution.uniform import sample_uniform
from kldm.kldm import ModelKLDM

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = resolve_data_root()

model = ModelKLDM(device=device).to(device)

root, device, model

ModuleNotFoundError: No module named 'src_kldm'

## 1. Load A Real DNG Batch

In [ ]:
batch = next(
    iter(DNGTask().dataloader(root=root, split='train', batch_size=2, shuffle=False, download=True))
).to(device)

print('num graphs :', batch.num_graphs)
print('pos shape  :', tuple(batch.pos.shape))
print('h shape    :', tuple(batch.h.shape))
print('l shape    :', tuple(batch.l.shape))
print('\nfirst 3 positions:')
print(batch.pos[:3])
print('\nfirst 3 one-hot atom rows:')
print(batch.h[:3])

num graphs : 2
pos shape  : (25, 3)
h shape    : (25, 118)
l shape    : (2, 6)

first 3 positions:
tensor([[6.6651e-01, 1.7850e-03, 3.3302e-01],
        [1.2300e-04, 9.9972e-01, 2.4600e-04],
        [3.3202e-01, 9.9566e-01, 6.6405e-01]])

first 3 one-hot atom rows:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.,

## 2. Sample Noisy KLDM Inputs

In [ ]:
t_graph = sample_uniform(lb=model.diffusion_l.eps, size=(batch.num_graphs, 1), device=device)
t_node = t_graph[batch.batch].squeeze(-1)

l_t, eps_l = model.diffusion_l.forward_sample(t=t_graph.squeeze(-1), x0=batch.l)
target_l = model.diffusion_l.score_target(t=t_graph.squeeze(-1), eps=eps_l)

a_t, eps_a = model.diffusion_a.forward_sample(t=t_node, x0=batch.h)
target_a = model.diffusion_a.score_target(t=t_node, eps=eps_a)

f_t, v_t, epsilon_v, epsilon_r, r_t = model.tdm.forward_sample(t=t_node, f0=batch.pos)
target_v = model.tdm.score_target(t=t_node, epsilon_v=epsilon_v, r_t=r_t, v_t=v_t)

print('t_graph shape:', tuple(t_graph.shape))
print('l_t shape    :', tuple(l_t.shape))
print('a_t shape    :', tuple(a_t.shape))
print('f_t shape    :', tuple(f_t.shape))
print('v_t shape    :', tuple(v_t.shape))
print('\nfirst 3 noisy positions:')
print(f_t[:3])
print('\nfirst 3 noisy atom rows:')
print(a_t[:3])
print('\nnoisy lattice vectors:')
print(l_t)

t_graph shape: (2, 1)
l_t shape    : (2, 6)
a_t shape    : (25, 118)
f_t shape    : (25, 3)
v_t shape    : (25, 3)

first 3 noisy positions:
tensor([[6.7163e-01, 1.1518e-02, 3.3647e-01],
        [7.2419e-04, 9.9166e-01, 2.3294e-03],
        [3.2321e-01, 5.4505e-03, 6.4863e-01]])

first 3 noisy atom rows:
tensor([[-2.0309e-01,  2.9005e-01,  1.6041e-01, -1.3560e-02, -1.6551e-01,
         -3.4667e-02,  7.6567e-02,  1.6329e-01,  4.1116e-01, -3.6297e-01,
          6.0578e-01,  3.4268e-01,  2.0042e-01, -2.6018e-01,  4.3901e-02,
         -2.9476e-01, -7.3997e-02, -2.4692e-01,  1.1361e-01, -9.6656e-02,
         -4.0641e-01,  5.9479e-02,  4.4588e-02,  4.9870e-02, -1.1339e-01,
          4.0828e-01,  2.9743e-01,  2.0804e-01,  1.1278e-01, -2.2556e-01,
          8.5792e-02,  4.7645e-02,  1.1061e-01, -2.7302e-01,  1.8731e-01,
         -3.6097e-01, -1.8295e-01,  6.8860e-03,  5.5925e-01, -4.8091e-02,
          1.8418e-01, -2.5764e-01,  3.8180e-01, -5.8101e-02, -2.1861e-02,
         -5.2887e-01,  1.335

## 3. Build The Real Score Network

In [ ]:
def denoise_score_matching_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    t: torch.Tensor,
    sigma_fn,
    match_dims,
    lambda_weight=None,
) -> torch.Tensor:
    """Use the same DSM form as in the continuous notebook.

    The loss is

        lambda(t) * || s_theta(x_t, t) - target ||^2

    with the default choice lambda(t) = sigma(t)^2.
    """
    if lambda_weight is None:
        lambda_weight = sigma_fn(t) ** 2
    weight = match_dims(lambda_weight, pred)
    return (weight * (pred - target) ** 2).mean()


def construct_velocity_score(
    pred_v: torch.Tensor,
    v_t: torch.Tensor,
    t: torch.Tensor,
    model: ModelKLDM,
) -> torch.Tensor:
    """Convert the network's velocity head into the KLDM-style score.

    This mirrors the paper structure:

        out_v = prefactor(t) * out_v - v_t / sigma_v(t)^2

    where t is first mapped to the internal TDM time interval.
    """
    diffusion = model.tdm
    t_internal = diffusion.time_scaling_T * t
    prefactor = (1.0 - torch.exp(-t_internal)) / (1.0 + torch.exp(-t_internal))
    prefactor = diffusion._match_dims(prefactor, pred_v)
    sigma_v_sq = diffusion._match_dims(diffusion.sigma_v(t_internal) ** 2, pred_v)
    return prefactor * pred_v - v_t / sigma_v_sq


optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model

ModelKLDM(
  (score_network): CSPVNet(
    (act_fn): SiLU()
    (node_embedding): Linear(in_features=118, out_features=128, bias=False)
    (atom_latent_emb): Linear(in_features=256, out_features=128, bias=True)
    (dis_emb): SinEmbedding()
    (time_emb): FourierEmbedding()
    (layers): ModuleList(
      (0-3): 4 x CSPVLayer(
        (dis_emb): SinEmbedding()
        (v_proj): Linear(in_features=3, out_features=60, bias=True)
        (edge_mlp): Sequential(
          (0): Linear(in_features=382, out_features=128, bias=True)
          (1): SiLU()
          (2): Linear(in_features=128, out_features=128, bias=True)
          (3): SiLU()
        )
        (node_mlp): Sequential(
          (0): Linear(in_features=256, out_features=128, bias=True)
          (1): SiLU()
          (2): Linear(in_features=128, out_features=128, bias=True)
          (3): SiLU()
        )
        (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
    )
    (final_layer_norm): LayerNorm

## 4. One Training Step

In [ ]:
batch = next(
    iter(DNGTask().dataloader(root=root, split='train', batch_size=8, shuffle=True, download=True))
).to(device)

t_graph = sample_uniform(lb=model.diffusion_l.eps, size=(batch.num_graphs, 1), device=device)
t_node = t_graph[batch.batch].squeeze(-1)

l_t, eps_l = model.diffusion_l.forward_sample(t=t_graph.squeeze(-1), x0=batch.l)
target_l = model.diffusion_l.score_target(t=t_graph.squeeze(-1), eps=eps_l)

a_t, eps_a = model.diffusion_a.forward_sample(t=t_node, x0=batch.h)
target_a = model.diffusion_a.score_target(t=t_node, eps=eps_a)

f_t, v_t, epsilon_v, epsilon_r, r_t = model.tdm.forward_sample(t=t_node, f0=batch.pos)
target_v = model.tdm.score_target(t=t_node, epsilon_v=epsilon_v, r_t=r_t, v_t=v_t)

scores = model.score_network(
    t=t_graph,
    pos=f_t,
    v=v_t,
    h=a_t,
    l=l_t,
    node_index=batch.batch,
    edge_node_index=batch.edge_node_index,
)

score_v = construct_velocity_score(scores['v'], v_t, t_node, model)
loss_v = denoise_score_matching_loss(score_v, target_v, model.tdm.time_scaling_T * t_node, model.tdm.sigma_v, model.tdm._match_dims)
loss_l = denoise_score_matching_loss(scores['l'], target_l, t_graph.squeeze(-1), model.diffusion_l.sigma, model.diffusion_l._match_dims)
loss_a = denoise_score_matching_loss(scores['h'], target_a, t_node, model.diffusion_a.sigma, model.diffusion_a._match_dims)
loss = loss_v + loss_l + loss_a

optimizer.zero_grad()
loss.backward()
optimizer.step()

print('score_v shape:', tuple(score_v.shape))
print('loss_v:', float(loss_v))
print('loss_l:', float(loss_l))
print('loss_a:', float(loss_a))
print('loss  :', float(loss))

score_v shape: (73, 3)
loss_v: 0.9396527409553528
loss_l: 1.2210992574691772
loss_a: 1.195951223373413
loss  : 3.356703281402588


## 5. Simple Training Loop

This is the same training pattern as the other notebooks: sample a batch, sample random times, build noisy inputs from the forward kernels, predict the scores, and regress against the exact targets.

In [ ]:
train_loader = DNGTask().dataloader(root=root, split='train', batch_size=16, shuffle=True, download=True)
num_epochs = 5
print_every = 20

for epoch in range(num_epochs):
    running_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)

        t_graph = sample_uniform(lb=model.diffusion_l.eps, size=(batch.num_graphs, 1), device=device)
        t_node = t_graph[batch.batch].squeeze(-1)

        l_t, eps_l = model.diffusion_l.forward_sample(t=t_graph.squeeze(-1), x0=batch.l)
        target_l = model.diffusion_l.score_target(t=t_graph.squeeze(-1), eps=eps_l)

        a_t, eps_a = model.diffusion_a.forward_sample(t=t_node, x0=batch.h)
        target_a = model.diffusion_a.score_target(t=t_node, eps=eps_a)

        f_t, v_t, epsilon_v, epsilon_r, r_t = model.tdm.forward_sample(t=t_node, f0=batch.pos)
        target_v = model.tdm.score_target(t=t_node, epsilon_v=epsilon_v, r_t=r_t, v_t=v_t)

        scores = model.score_network(
            t=t_graph,
            pos=f_t,
            v=v_t,
            h=a_t,
            l=l_t,
            node_index=batch.batch,
            edge_node_index=batch.edge_node_index,
        )

        score_v = construct_velocity_score(scores['v'], v_t, t_node, model)
        loss_v = denoise_score_matching_loss(score_v, target_v, model.tdm.time_scaling_T * t_node, model.tdm.sigma_v, model.tdm._match_dims)
        loss_l = denoise_score_matching_loss(scores['l'], target_l, t_graph.squeeze(-1), model.diffusion_l.sigma, model.diffusion_l._match_dims)
        loss_a = denoise_score_matching_loss(scores['h'], target_a, t_node, model.diffusion_a.sigma, model.diffusion_a._match_dims)
        loss = loss_v + loss_l + loss_a

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss)

        if step % print_every == 0:
            print(
                f'epoch={epoch:02d} step={step:03d} '
                f'loss={float(loss):.6f} '
                f'(v={float(loss_v):.4f}, l={float(loss_l):.4f}, a={float(loss_a):.4f})'
            )

    mean_loss = running_loss / (step + 1)
    print(f'epoch={epoch:02d} mean_loss={mean_loss:.6f}')

NameError: name 'root' is not defined